In [1]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

# Load env variables and connect to database
load_dotenv("../.env")
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DATABASE_URL)

# Read engineered data
df = pd.read_sql("SELECT * FROM engineered_housing_data", engine)

# Target has large values → apply log transform to make learning easier
# later we will convert predictions back using exp
df['sale_price'] = np.log1p(df['sale_price'])

# Split features and target
X = df.drop('sale_price', axis=1)
y = df['sale_price']

# Split data: 80% train, 20% test
# random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Save splits to database
# target values are already log-transformed
X_train.to_sql('X_train', engine, if_exists='replace', index=False)
X_test.to_sql('X_test', engine, if_exists='replace', index=False)
y_train.to_frame().to_sql('y_train', engine, if_exists='replace', index=False)
y_test.to_frame().to_sql('y_test', engine, if_exists='replace', index=False)

print(f"Train-test split done. Train: {len(X_train)}, Test: {len(X_test)}")

Train-test split done. Train: 2344, Test: 586
